# Final Push

In [ ]:
# debug mode flag
DEBUG_MODE = True # TODO doublecheck output/logging information when debug is disabled, ensure only the actual important bits is outputted, skip the nitty gritty details of the debug level logging for prod env

# TODO low priority: consider making a simple wrapper class for the dumb ANSI escape sequences so they're more readable in the source, or going full speed into rich or colorama territory
if DEBUG_MODE:
    print(
        f"\033[31m WARNING: DEBUG MODE is ON!!! \033[0m Verbose output and debug logging enabled."
    )


In [ ]:
# user configurable settings, there is an additional configurable settings block after the WhisperX auto-config portion with model-specific settings such as thread count, batch size, compute datatype, toggle-able diarization, alignment, all those goodies etc.

# toggle offline mode on/off, needs to be run in Online mode (set `os.environ["HF_HUB_OFFLINE"] = "0"`) for at least the first run of the application in order to download and cache the models locally. Gated HF models are an annoyance ARGH # TODO this should be relatively simple to implement an "online mode for first run to retrieve the needed models, verify their existence in `MODEL_DIR`, then toggle to offline mode for subsequent runs and run off _only_ cached local models in `MODEL_DIR`" automated functionality (less user choice = less user confusion = less error opportunities = better user experience = less emergency calls to the tech support line for dumb issues = less annoying tech support calls in general = all around win)

import os # TODO tidy imports, random one-offs all over the place
os.environ["HF_HUB_OFFLINE"] = "0"  # "1" for on (Offline mode enabled), "0" for off (Online mode enabled, will download models from HF using API token and cache them locally in MODEL_DIR below)

# note: default cached models location for WhisperX models is C:\Users\<username>\.cache\huggingface\hub, # TODO update this to a directory within our application structure, app/model_cache or something?
MODEL_DIR = "model_cache/" # for now # TODO doublecheck that all the models are in sync with this as the cached model location, ensure that this is project root relative path

# hardcoded settings that will be # TODO refactor into parameters of respective class methods later, to be updated when merged and refactored into main project structure
OUTPUT_FILE = "transcript.txt"  # Output file written to the cwd for now

INPUT_FILE = "sample_data/en/11_Lecture-48k.wav"  # should accept basically any audio or video file that ffmpeg can detect audio in, mp4 or otherwise, refer to ffmpeg docs for more info, or fire up CLI and run `ffmpeg -decoders` for supported decoders to convert source input file to individual basic-datatype-based (array of pixels/bitmap style representation for video (not relevant to us), PCM samples for audio) frames for processing, `ffmpeg -demuxers` for the supported demuxers to extract container data, `ffmpeg -formats` for a list of supported formats, or `ffmpeg -codecs` for a full list of supported codecs, which amounts to a wide array of supported formats/codecs/demuxers/decoders/etc. Refer to FFmpeg docs for further details. I can't say that it will handle _everything_ out there, but pretty damn close for our purposes...
# TODO maybe? low priority, wrapper function to output a formatted string representing the list of acceptable and supported audio formats/codecs/etc to display to the user, probably keep it as a key/value dict type for likely use with a select input in front-end etc


env_file = ".env" # environment specific settings, note to self: for love of all that is holy, do not commit this file to repo! ever! lest ye sins be purged with fire! Just update the .env.example file with variable name, set to nothing (`MY_VAR=`), and commit that file, and communicate sensitive environment vars like API keys, db credentials in a more secure channel, like discord, lol and dump them in your local .env file (reminder: never to be committed and especially never committed and pushed to public repo! always to be in a .gitignore file somewhere!). 



In [ ]:
# main setup/import/general settings cell

# TODO need to prune unused imports, organize, isort or similar to make it easier to parse for us mere mortals (and maybe even our future selves in a month's time... /foreshadowing and ominous Dies irae theme music plays https://www.youtube.com/watch?v=bTY3s4kvc8k ), etc
import logging, time, subprocess, re, sys, json, time, os
from datetime import datetime, timedelta
from pathlib import Path
import whisperx
from faster_whisper import WhisperModel
from pydantic_settings import BaseSettings, SettingsConfigDict
import numpy as np

# basic logging setup, default handler outputs to stdout already, # TODO maybe add a file output handler to log to a local log file (temp file, etc), or log records to the db? What schema for log record table? Timestamp, exception class name, exception message text, maybe a join'd traceback string too? ehhhhh, not important right now. Will resort to additional logging behaviors as needed, /if/ needed...

logging.basicConfig(level=logging.INFO if not DEBUG_MODE else logging.DEBUG)  # toggle debug output here
logging.info(f"Basic logger setup!")

start_time = time.perf_counter()  # begin timer for determining final execution time at end, also double-purposing it to determine execution time for the setup, config, hardware and software detection, etc steps that follow shortly here # TODO perhaps a better, neater way to time code execution? timeit or similar? or just be lazy and continue using time.perf_counter() because its so straightforward and doesn't add any significant overhead/dependencies?
logging.info(f"Execution officially starting! Start time: {datetime.now()}")

if os.environ["HF_HUB_OFFLINE"] == "1":
    logging.info(
        # blue text indicating offline mode active
        f"\033[94m Offline mode set. \033[0m Will use local (cached) models only." # # TODO FIXME "allegedly" local only, but somehow urllib3 is processing HTTPS requests according to the debug log... but offline mode is enabled... maybe there is a dependency that is still phoning home for model information and needs some quick and effective STFU treatment?investigate me! the hell? Stahp, urllib...... STAHP!
    )
else:
    logging.warning(
        # red warning text indicating to user that online mode is ACTIVE, and will require network connection for initial downloading of models to a local cache
        f"\033[31m Warning! Online mode set! \033[0m This will be _slow_ (internet download speed dependent)... first run for models automatically selected for my craptop specs (integrated intel uhd graphics, no gpu, no cuda, just cpu and 8gb of RAM to work with, hopefully representing the far lower bound for specs for endusers... 'hopefully'...)) for first run, but once models have been downloaded and cached locally, set `os.environ['HF_HUB_OFFLINE'] = \"1\"` in the designated config above, this env variable is set in the initial user configurable settings chunk) to disable online mode and use only the local (cached) models (network connection/internet access should _not_ be required after that point)."
    )

# performance reference notes: testing dev laptop specs:
# CPU: Intel Core i3 1215U Alder Lake 10nm Technology
# RAM: 8.00GB Single-Channel DDR4 @ 1330MHz (19-19-19-43)
# Graphics: Generic PnP Monitor (1920x1080@120Hz)
# 		    Intel UHD Graphics (Dell) # fffffuuuuuuuuu
# Storage: 238GB NVMe PM9C1a Samsung 256GB (Unknown (SSD))
# as mentioned, I hope... I really really hope... that this represents the far low end of the system spec range our endusers will have access to


# setup .env settings obj
class GlobalAppSettings(BaseSettings):
    hf_token: str # HuggingFace API token, required for downloading of gated pyannote models, but only needed the first time to download and cache the models. # TODO though, this brings up the point that if a user changes hardware specs, they will have an unexpected side effect if the changed hardware influences the autoconfig model selection, potentially selecting a model that /isn't/ already cached from the initial first run/install...
    # TODO aggregate all env vars here (well, in the ./.env file anyway), db keys, credentials, general app config, model parameter config, transcript format settings, and all that jazz...
    model_config = SettingsConfigDict(
        env_file=".env", # default env file
        env_file_encoding="utf-8",
        extra="allow",  # `extra` param allows adding env vars after init, lazy mode
    )


# when merged into main, settings will likely be accessible in a a "global-esque" variable (Bjarne forgive me) living inside the main_window class # TODO potentially shift to storing global references like this in the main application class? There doesn't seem to be a strong consensus online either way... ehhhhh, not important right now. Will fallback to "if it ain't broke, don't fix it"

settings = GlobalAppSettings()
logging.info(f"Pydantic Global App Settings object initialized and set to {env_file}.")


INFO:root:Basic logger setup!
INFO:root:Execution officially starting! Start time: 2026-03-09 12:06:31.573264


NameError: name 'env_file' is not defined

In [48]:
import torch
import torchaudio
import pyannote.audio

if DEBUG_MODE:
    logging.info(f"Checking versions of torch, torchaudio, and pyannote.audio...")
    logging.info(f"{torch.__version__=}")
    logging.info(f"{torchaudio.__version__=}")
    logging.info(f"{pyannote.audio.__version__=}")

# torchcodec is still not playing nicely with the various other libraries, but we've since bypassed the requirement for it utilizing ffmpeg to decode audio, so these warnings and errors are expected and can be safely ignored without issue. Nothing to see here, folks


INFO:root:Checking versions of torch, torchaudio, and pyannote.audio...
INFO:root:torch.__version__='2.8.0+cpu'
INFO:root:torchaudio.__version__='2.8.0+cpu'
INFO:root:pyannote.audio.__version__='4.0.4'


In [49]:
# attempt to automatically determine optimal whisperx config based on system specs

# noteworthy config settings to tweak as needed: # TODO add these to the GlobalAppSettings object
prefer_accuracy = True  # for details for both of these params, see docstring for the auto_configure_whisperx function below
prefer_speed = False

# this entire chunk started as pseudocode notes and a skeleton from Claude, and has been mangled enough to where I can't even recognize the original code anymore, likely needs a solid cleanup and refactor, move imports to top of file, etc etc
import os
import sys
import platform
import psutil
import subprocess
from dataclasses import dataclass, field
from typing import Optional


@dataclass
class WhisperXConfig:
    model_size: str
    device: str
    compute_type: str
    batch_size: int
    device_index: int = 0
    threads: int = 4

    # Metadata for debugging/logging, very useful
    _reason: dict = field(default_factory=dict, repr=False)

    def to_dict(self) -> dict:
        return {
            "model_size": self.model_size,
            "device": self.device,
            "compute_type": self.compute_type,
            "batch_size": self.batch_size,
            "device_index": self.device_index,
            "threads": self.threads,
        }

    def explain(self) -> str:
        lines = ["WhisperX Config Rationale:"]
        for k, v in self._reason.items():
            lines.append(f"  {k}: {v}")
            
        return "\n".join(lines)


def _get_cuda_info() -> tuple[bool, Optional[int], Optional[str]]:
    """
    Returns (cuda_available, vram_mb, gpu_name).
    Tries torch first, falls back to nvidia-smi.
    """

    # Try PyTorch first...
    try:
        import torch

        if torch.cuda.is_available():
            idx = torch.cuda.current_device()
            vram = torch.cuda.get_device_properties(idx).total_memory // (1024**2)
            name = torch.cuda.get_device_name(idx)
            return True, vram, name

    except ImportError:
        # not handling here, just letting execution continue and return False etc
        pass

    # Fallback: trying nvidia-smi...
    try:
        result = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=memory.total,name",
                "--format=csv,noheader,nounits",
            ],
            capture_output=True,
            text=True,
            timeout=5,
        )

        if result.returncode == 0:
            parts = result.stdout.strip().split(",")
            vram = int(parts[0].strip())
            name = parts[1].strip() if len(parts) > 1 else "Unknown GPU"
            return True, vram, name

    except (FileNotFoundError, subprocess.TimeoutExpired, ValueError) as e:
        # not worrying about these exceptions here, just letting execution continue and return False etc
        pass

    return False, None, None


def _get_ram_gb() -> float:
    return psutil.virtual_memory().total / (1024**3)


def _get_cpu_cores() -> int:
    return (
        psutil.cpu_count(logical=False) or psutil.cpu_count(logical=True) or 2
    )  # cascading fallback to default of 2 cores


def _supports_float16_cpu() -> bool:
    """Check if CPU supports float16 via AVX2 (rough heuristic)."""

    try:
        import subprocess

        if platform.system() == "Linux":
            result = subprocess.run(
                ["grep", "-m1", "avx2", "/proc/cpuinfo"], capture_output=True, text=True
            )
            return "avx2" in result.stdout
        
        elif platform.system() == "Darwin":
            result = subprocess.run(
                ["sysctl", "-n", "machdep.cpu.features"], capture_output=True, text=True
            )
            
            return "AVX2" in result.stdout

    except Exception as e:
        # not handling here, just letting execution continue and return False etc
        pass

    return False


def auto_configure_whisperx(
    prefer_accuracy: bool = True,
    prefer_speed: bool = False,
    verbose: bool = False,
) -> WhisperXConfig:
    """
    Attempts to automatically determine optimal WhisperX configuration based on current system specs.

    Args:
        prefer_accuracy: Bias toward larger models even at cost of speed.
        prefer_speed:    Bias toward smaller models / faster compute types.
        verbose:         Print rationale to stdout. (highly suggested)

    Returns:
        WhisperXConfig dataclass with recommended default WhisperX model settings based on system specs.
    """

    reasons = {}

    # ── Hardware detection ───────────────────────────────────────────────────
    cuda_available, vram_mb, gpu_name = _get_cuda_info()
    ram_gb = _get_ram_gb()
    cpu_cores = _get_cpu_cores()
    cpu_float16 = _supports_float16_cpu()

    reasons["ram_gb"] = f"{ram_gb:.1f} GB"
    reasons["cpu_cores"] = cpu_cores
    reasons["cuda"] = (
        f"{cuda_available} ({gpu_name}, {vram_mb} MB VRAM)"
        if cuda_available
        else "not available"
    )

    # ── Device ───────────────────────────────────────────────────────────────
    if cuda_available and vram_mb is not None and vram_mb >= 2048:
        device = "cuda"
        reasons["device"] = "CUDA selected (GPU detected with sufficient VRAM)"
    else:
        device = "cpu"
        reasons["device"] = "CPU selected (no CUDA or insufficient VRAM)"

    # ── Model size ───────────────────────────────────────────────────────────
    # Reference:
    # Approximate VRAM / RAM requirements per model:
    #   tiny   ~1 GB   large-v2 ~10 GB
    #   base   ~1 GB   large-v3 ~10 GB
    #   small  ~2 GB   turbo    ~6 GB
    #   medium ~5 GB
    if device == "cuda":
        resource = vram_mb / 1024  # GB
        resource_label = f"{resource:.1f} GB VRAM"
    else:
        resource = ram_gb
        resource_label = f"{resource:.1f} GB RAM"

    # Apply bias modifiers
    bias = 1 if prefer_accuracy else (-1 if prefer_speed else 0)

    model_tiers = [
        (1.5, "tiny"),
        (2.5, "base"),
        (4.0, "small"),
        (7.0, "medium"),
        (11.0, "large-v2"),
        (
            float("inf"),
            "large-v3",
        ),  # for anything above the 11.0 threshold, use large-v3
    ]

    chosen_model = "tiny"
    for threshold, name in model_tiers:
        if resource >= threshold:
            chosen_model = name

    # Apply bias: shift one tier up or down
    model_names = [m for _, m in model_tiers]
    idx = model_names.index(chosen_model)
    idx = max(0, min(len(model_names) - 1, idx + bias))
    chosen_model = model_names[idx]

    reasons["model_size"] = f"{chosen_model} (based on {resource_label})"

    # ── Compute type ─────────────────────────────────────────────────────────
    if device == "cuda":
        # float16 is standard on modern NVIDIA; use int8 only on very low VRAM
        if vram_mb >= 4096:
            compute_type = "float16"
            reasons["compute_type"] = "float16 (CUDA, ≥4 GB VRAM)"
        else:
            compute_type = "int8_float16"
            reasons["compute_type"] = "int8_float16 (CUDA, limited VRAM)"
    else:
        # CPU: int8 is fastest and well-supported; float16 rarely beneficial on CPU
        if prefer_accuracy and cpu_float16:
            compute_type = "float32"
            reasons["compute_type"] = "float32 (CPU, accuracy mode, AVX2 present)"
        else:
            compute_type = "int8"
            reasons["compute_type"] = "int8 (CPU default — best speed/memory tradeoff)"

    # ── Batch size ───────────────────────────────────────────────────────────
    if device == "cuda":
        if vram_mb >= 10000:
            batch_size = 32
        elif vram_mb >= 6000:
            batch_size = 16
        elif vram_mb >= 4000:
            batch_size = 8
        else:
            batch_size = 4
    else:
        # CPU batch sizing based on RAM and cores
        if ram_gb >= 32 and cpu_cores >= 8:
            batch_size = 8
        elif ram_gb >= 16 and cpu_cores >= 4:
            batch_size = 4
        else:
            batch_size = 2

    if prefer_speed:
        batch_size = min(batch_size * 2, 64)
        reasons["batch_size"] = f"{batch_size} (speed-biased, doubled)"
    elif prefer_accuracy:
        batch_size = max(batch_size // 2, 1)
        reasons["batch_size"] = f"{batch_size} (accuracy-biased, halved)"
    else:
        reasons["batch_size"] = str(batch_size)

    # ── CPU threads ──────────────────────────────────────────────────────────
    # WhisperX passes this to faster-whisper / CTranslate2
    threads = max(2, min(cpu_cores, 8))  # cap at 8; diminishing returns beyond that
    reasons["threads"] = f"{threads} (of {cpu_cores} physical cores)"

    # ── Build config ─────────────────────────────────────────────────────────
    config = WhisperXConfig(
        model_size=chosen_model,
        device=device,
        compute_type=compute_type,
        batch_size=batch_size,
        device_index=0,
        threads=threads,
        _reason=reasons,
    )

    if verbose:
        logging.info(config.explain())

    return config


In [50]:
whisperx_cfg = auto_configure_whisperx(
    verbose=True,
    prefer_speed=prefer_speed,
    prefer_accuracy=prefer_accuracy
)

logging.info(
    f"WhisperX config has been automatically set based on current system specs!"
)


INFO:root:WhisperX Config Rationale:
  ram_gb: 31.7 GB
  cpu_cores: 12
  cuda: True (NVIDIA T1000, 4096 MB VRAM)
  device: CUDA selected (GPU detected with sufficient VRAM)
  model_size: medium (based on 4.0 GB VRAM)
  compute_type: float16 (CUDA, ≥4 GB VRAM)
  batch_size: 4 (accuracy-biased, halved)
  threads: 8 (of 12 physical cores)
INFO:root:WhisperX config has been automatically set based on current system specs!


In [ ]:
# user configurable settings part deux
# most are automatically set via auto_configure_whisperx() above based on system specs, but can be overridden manually here
# change these at your own risk!

logging.info(f"Executing any manual overrides to auto-configured WhisperX model settings...")

# quick reference complements of le Google for the various WhisperX/faster_whisper-based variants, models and sizes:
# Available Model Sizes in WhisperX:
#     tiny: The smallest and fastest models, with the lowest accuracy.
#         Parameters: 39 M
#         English-only: tiny.en
#         Multilingual: tiny
#     base: A small step up in accuracy and size.
#         Parameters: 74 M
#         English-only: base.en
#         Multilingual: base
#     small: A good balance of speed and medium accuracy.
#         Parameters: 244 M
#         English-only: small.en
#         Multilingual: small
#     medium: Higher accuracy for more complex audio, with increased resource needs.
#         Parameters: 769 M
#         English-only: medium.en
#         Multilingual: medium
#     large: The most accurate models, requiring the most VRAM and processing time.
#         Parameters: 1550 M
#         Multilingual: large, large-v1, large-v2, large-v3
#     turbo: An optimized version of large-v3 offering significant speed improvements with comparable accuracy.
#         Parameters: 809 M
#         Multilingual: turbo or large-v3-turbo
# 
# WhisperX model size names quick ref:
#     tiny, tiny.en
#     base, base.en
#     small, small.en
#     medium, medium.en
#     large, large-v1, large-v2, large-v3, large-v3-turbo
#     Distilled models: distil-large-v2, distil-medium.en, distil-small.en, distil-large-v3 # more info on distil models on HF (sliding window instead of 30s segments): https://huggingface.co/distil-whisper/distil-large-v3
#     Custom models: nyrahealth/faster_CrisperWhisper
#                    NOTE: see https://huggingface.co/nyrahealth/faster_CrisperWhisper for more info on this specially optimized model. 
#                          Originally based on CrisperWhisper model for "crisp" word-level aligned timestamps, then optimized for using 
#                          CTranslate2 on the backend for performance, and focusing on high-accuracy word-level aligned timestamps, claiming: 
#                              🎯 Accurate Word-Level Timestamps: Provides precise timestamps, even around disfluencies and pauses, by utilizing an adjusted tokenizer and a custom attention loss during training.
#                             📝 Verbatim Transcription: Transcribes every spoken word exactly as it is, including and differentiating fillers like "um" and "uh".
#                             🔍 Filler Detection: Detects and accurately transcribes fillers.
#                             🛡️ Hallucination Mitigation: Minimizes transcription hallucinations to enhance accuracy.
# TODO even upon skimming docs, this custom "crisp"-focused model and its specific accuracy-focused (with the goal of achieving as _close_ to **verbatim**-level transcription as possible) features seem like it may have merit and worth a more thorough review, given our criteria for high accuracy
# # (which is very likely more than we'd ever need, but can be an option if ever needed, especially if desired accuracy goals are not met with WhisperX's "less-optimized for verbatim-level transcription" faster_whisper-based models)
# # TODO we don't have an absolute way to measure and benchmark these just yet, need to research more on WER word error rate and benchmarking metrics for accuracy for transcription, translation, and diarization (if such a metric for diarization even exists without a heavily-curated toy example dataset, that would be great... )



# Random relevant reference links:


# https://huggingface.co/distil-whisper/distil-large-v3 has a great quantity of detailed aspects of the various distil models and how they measure up against vanilla openai/whisper, whisper cpp, etc
# also has a section about audio datasets used for evaluating the models' performance and accuracy, https://huggingface.co/distil-whisper/distil-large-v3#evaluation # TODO definitely re-read over this
# and has a nice table of the various training data sources used for training the models https://huggingface.co/distil-whisper/distil-large-v3#data
# MIT licensed
# distil-whisper academic paper: https://arxiv.org/abs/2311.00430

# TODO read me and learn! much learning to be had here: https://huggingface.co/blog/whisper-speculative-decoding https://huggingface.co/blog/whisper-speculative-decoding#baseline-implementation

# TODO read and learn (separate the transcription LLM call and the translation LLM call): https://huggingface.co/openai/whisper-large-v2#model-details


# So far, I have been quite unsuccessful in seeking out (free and usable given licensing constraints) well-documented and human-curated audio datasets and associated verbatim (human-verified) transcripts to match
# 
# https://huggingface.co/datasets/openslr/librispeech_asr CURATED DATASETS FUCK YES
# https://www.openslr.org/12 alternate
# curated dataset on kagglehub! https://www.kaggle.com/datasets/pypiahmad/librispeech-asr-corpus
# more audio dataset information: https://huggingface.co/blog/audio-datasets#a-complete-guide-to-audio-datasets

# TODO READ ME Pipelines 101: https://huggingface.co/docs/transformers/en/pipeline_tutorial



# TODO research language specific checkpoints and different implementations of base whisper models: https://huggingface.co/models?other=whisper

# probably unneeded, but interesting, functions to convert models between CTranslate2 model types: https://opennmt.net/CTranslate2/python/ctranslate2.converters.TransformersConverter.html






# basic Tips for File Processing in base Whisper
# https://guides.libraries.emory.edu/c.php?g=1442123&p=10711508#

# metrics for accuracy of whisper transcription/translation/diarization and such:
# https://github.com/analyticsinmotion/werpy - werpy library for calculating and analyzing the Word Error Rate (WER) for speech recognition tasks, comparing between two sets of misaligned text
# https://jitsi.github.io/jiwer/ - jiwer library for: 
    # word error rate (WER)
    # match error rate (MER)
    # word information lost (WIL)
    # word information preserved (WIP)
    # character error rate (CER)
#   These measures are computed with the use of the minimum-edit distance between one or more reference and hypothesis sentences. The minimum-edit distance is calculated using RapidFuzz, which uses C++ under the hood, and is therefore faster than a pure python implementation.

# WER implementation from scratch: https://thepythoncode.com/article/calculate-word-error-rate-in-python

# https://github.com/Picovoice/speech-to-text-benchmark

# Whisper Large V3 Speech Recognition Benchmark: 1 Million hours of audio transcription for just $5110  https://blog.salad.com/whisper-large-v3/

# base whisper benchmark comparison over GPUs: https://github.com/openai/whisper/discussions/918

# https://medium.com/@bnjmn_marie/whisper-large-v3-turbo-as-good-as-large-v2-but-6x-faster-97f0803fa933

# TODO READ AND LEARN: https://cdn.openai.com/papers/whisper.pdf

# TODO potential viable altnerative? https://github.com/moonshine-ai/moonshine/tree/main?tab=readme-ov-file#available-models
# https://github.com/moonshine-ai/moonshine?tab=readme-ov-file#when-should-you-choose-moonshine-over-whisper

# faster-whisper is a reimplementation of OpenAI's Whisper model using CTranslate2, which is up to 4x times faster than openai/whisper for the same accuracy while using less memory. The efficiency can be further improved with 8-bit quantization on both CPU and GPU.
# It can automatically detect the following 14 languages and transcribe the text into their respective languages: en, zh, fr, de, ja, ko, ru, es, th, it, pt, vi, ar, tr. https://github.com/SYSTRAN/faster-whisper
# langchain FasterWhisperParser docs: https://reference.langchain.com/python/langchain-community/document_loaders/parsers/audio/FasterWhisperParser
# faster_whisper also supports batched transcription, which can be a major perk if we want to handle a large number of audio transcription, translation, diarization, summarization, or other audio-related tasks in a more parallelizable fashion
# https://github.com/SYSTRAN/faster-whisper#batched-transcription # TODO research VAD config options for finetuning

# https://medium.com/@balaragavesh/converting-your-fine-tuned-whisper-model-to-faster-whisper-using-ctranslate2-b272063d3204

# TODO useful? https://www.union.ai/blog-post/parallel-audio-transcription-using-whisper-jax-and-flyte-map-tasks-for-streamlined-batch-inference

# TODO possible container solution? https://github.com/basetenlabs/truss

# https://docs.baseten.co/quickstart

# https://pypi.org/project/SpeechRecognition/

# TODO read me https://opencv.org/applications-of-speech-recognition/

# offtopic https://www.automl.org/automl/

# TODO read https://charanhu.medium.com/how-to-inspect-a-huggingface-model-architecture-in-python-a-complete-guide-5ba6d1963231#:~:text=Prerequisites%20and%20Setup%20For%20this%20tutorial,%20we%27ll,models%20will%20be%20downloaded%20and%20cached%20locally.

# unrelated, image/object detection, etc: https://github.com/ultralytics/yolov5

# cached/local model locations: https://github.com/huggingface/diffusers/discussions/6675
# similar: cached local model dir override: https://github.com/m-bain/whisperX/issues/425
# similar: https://github.com/m-bain/whisperX/issues/337

# https://www.linkedin.com/pulse/managing-local-llms-make-boring-xiao-fei-zhang-wbrhe


# interesting # TODO read this https://www.parseable.com/blog/baseten-monitoring-setup

# interesting, not so related: https://www.linkedin.com/posts/matthew-hallett-041a3881_ive-just-released-a-full-beginners-guide-activity-7344796586696183810-Ms7F

# TODO interesting https://www.nvidia.com/en-us/case-studies/baseten-cloud-scaling-ai-inference/

# TODO cannabalize and assimilate knowledge from https://www.baseten.co/resources/type/guide/?gad_source=1&gad_campaignid=21493664768

# baseten deployment vid tutorial https://www.youtube.com/watch?v=Zl5frYNmHw8


# TODO read https://docs.baseten.co/development/chain/overview

# https://huggingface.co/openai/whisper-large-v3-turbo

# a long list of community forks based on faster_whisper, wide variety of specialized use cases and finetuned models:
# https://github.com/SYSTRAN/faster-whisper#community-integrations

# faster_whisper performance comparison notes
# https://github.com/SYSTRAN/faster-whisper#comparing-performance-against-other-implementations

# finetuning whisper model tips for Dutch transcription/translation optimization
# https://blog.ml6.eu/fine-tuning-whisper-for-dutch-language-the-crucial-role-of-size-dd5a7012d45f
# similar, but for general foreign languages: https://huggingface.co/blog/fine-tune-whisper

# TODO is this useful? https://huggingface.co/Mozilla/whisperfile

# ffmpeg related: FFmpeg in Production: Codecs, Performance, and Licensing - https://getstream.io/blog/ffmpeg/
# https://img.ly/blog/ultimate-guide-to-ffmpeg/

# choosing whisper variants: https://modal.com/blog/choosing-whisper-variants


# TODO read for Networks class: https://mixtape.scunning.com/
# TODO me too! https://visone.ethz.ch/wiki/index.php/Backbone_Layout
# https://cran.r-project.org/web/packages/backbone/vignettes/backbone.html

# interesting dev/backend article from Baseten about optimizing base whisper models for stupidly significant speed improvements, on the order of up to 2400x (!), **AND** performing the speech-to-text process in faster-than-real-time... (but it does appear to be cloud-based and commercial, so not free... and likely paired with a restrictive commercial use license ie. that's definitely a big ole negatory on that one, blue leader) still, # TODO second look at potentials and possibilities using Baseten's models? 
# Generally Available: The fastest, most accurate and cost-efficient Whisper transcription. At Baseten, we've built the most performant (2400x real-time factor), accurate, and cost-efficient speech-to-text pipeline for production AI audio transcription 
# https://www.baseten.co/blog/the-fastest-most-accurate-and-cost-efficient-whisper-transcription/

# more from Baseten, including cost benchmark comparison among the larger v2/v3-sized models, diarization accuracy metrics comparison (based on Diarization Error Rate (DER)), 
# https://www.baseten.co/blog/the-fastest-whisper-transcription-with-streaming-and-diarization/


# general model size note: if source audio language is all English, can optimize performance significantly by using the .en English-only model variants (tiny.en, base.en, small.en, medium.en, whatever) resulting in a smaller model size _and_ faster transcription processing, win win
MODEL_SIZE = (
    whisperx_cfg.model_size
) # WhisperX/faster_whisper model sizes: tiny | base | small | medium | large-v2 | large-v3 (see https://github.com/guillaumekln/faster-whisper#available-models for more re: faster_whisper, and )

# None for language auto-detection, or... 
# 2 digit ISO 639 language code for the spoken language in input audio file (without locale) such as "en" for English. Reference wiki list of ISO 639 language codes: https://en.wikipedia.org/wiki/List_of_ISO_639_language_codes
# note: WhisperX models need minimum >30s of audio to even attempt a potshot at detecting the language spoken, _on the entire transcribed audio, and has significant difficulties with multilingual audio in any form or combination, compounded by constraint of enforced lack of internet access and local-only model usage (after initial download of cached models)
LANGUAGE = None  

DEVICE = whisperx_cfg.device # "cpu" or "cuda"

COMPUTE_TYPE = (
    whisperx_cfg.compute_type
)  # "int8" (CPU-friendly) | "float16" (GPU) | "float32"

BATCH_SIZE = 16 # Reduce if you hit OOM on GPU; safe to leave as-is for CPU

ALIGN_OUTPUT = ( # unneeded at this point, resource intensive and time/compute/token usage noticably goes up
    False  # Word-level alignment (requires internet access for first run to download and cache models locally)
)

THREADS = whisperx_cfg.threads # Number of threads to use for transcription

DEVICE_INDEX = whisperx_cfg.device_index # Index of GPU device to use

HF_TOKEN = settings.hf_token  # HuggingFace token — only needed for speaker diarization gated model (located in the .env file in project root)
DIARIZE = True  # Speaker diarization (requires HF_TOKEN set)


INFO:root:Executing any manual overrides to auto-configured WhisperX model settings...


In [ ]:
logging.info(f"Configuring path for FFMpeg and determining installed FFMpeg version...")

# attempts to get the version of the ffmpeg executable, assuming it's found in the sys.PATH somewhere
def get_ffmpeg_version() -> str:
    """
    Returns the ffmpeg version string (e.g., '6.1.1').

    Raises:
        RuntimeError: if ffmpeg is not found in path, not installed or version cannot be parsed.
    """

    try:
        result = subprocess.run(
            ["ffmpeg", "-version"], check=True, capture_output=True, text=True
        )

    except FileNotFoundError as e:
        logging.error(
            f"\033[31m ERROR: ffmpeg is not installed or not found in PATH: \033[0m {e}"
        )
        raise RuntimeError("ffmpeg is not installed or not found in PATH.")

    except subprocess.CalledProcessError as e:
        logging.error(f"\033[31m ERROR: ffmpeg returned an error: \033[0m {e}")
        raise RuntimeError(f"ffmpeg returned an error: {e.stderr}") from e

    # Typical first line looks like:
    # "ffmpeg version n6.1.1-3-g123abc blah blah stuff this version was built with, blah blah blah"
    # second and onward lines have build info and other details, not needed for our purposes
    first_line = result.stdout.splitlines()[0]

    match = re.search(r"ffmpeg version\s+([^\s]+)", first_line)
    if not match:
        logging.error(
            f"\033[31m ERROR: Could not parse ffmpeg version from output's first line: \033[0m \"{first_line}\""
        )
        raise RuntimeError("Could not parse ffmpeg version output.")

    version = match.group(1)

    # Optional: strip leading 'n' sometimes present in builds (e.g., n6.1.1)
    version = version.lstrip("n")

    return version


def print_formatted_sys_path():
    # Rather than deal with the raw output of sys.PATH, we can use the json module to spit out a nicely formatted list of the current sys.PATHs
    formatted_path = json.dumps(sys.path, indent=4)

    # use some colors to make it pretty!
    # not gonna lie, I asked Claude for the ANSI codes for changing terminal output color, without shame. This simple use didn't warrant importing a full proper terminal color package like rich or colorama, etc
    logging.info("\033[1;34m [sys.PATH] \033[0m")  # ANSI code for blue for the title
    logging.info(
        "\033[1;32m" + formatted_path + "\033[0m"
    )  # ANSI code for green for paths, then ANSI code for reset


if DEBUG_MODE:
    logging.info(f"Outputting current sys.PATH:")
    print_formatted_sys_path()

try:
    ffmpeg_version = get_ffmpeg_version()

except Exception as e:
    logging.error(f"\033[31m General Recoverable ERROR: Could not find ffmpeg version: \033[0m {e}")
    # this is easily recoverable, so I'm choosing to not re-raise an exception, but instead just print it, and default to adding the local ffmpeg bin path to sys.PATH
    # raise RuntimeError("ffmpeg not found in path.")

    # ffmpeg not found in path => add local ffmpeg path (in project root) to path
    logging.info(f"Attempting to add local ffmpeg path to sys.PATH...")
    parent_dir = Path.cwd().resolve().parent  # parent = project root, currently running from test_notebooks dir, one folder down from project root, will need update when merged to main # TODO update this when merged to main
    ffmpeg_bin_path = (
        parent_dir / "ffmpeg" / "bin"
    )  # will look for local ffmpeg here, as well as the rest of sys.PATH

    if ffmpeg_bin_path.exists():
        sys.path.append(str(ffmpeg_bin_path))
        logging.info(
            f"Directory `{ffmpeg_bin_path}` found and has been added to sys.PATH!"
        )
        
        logging.info(f"Now that we've recovered from not finding ffmpeg in the sys.PATH, need to retry getting the version...")
        try: # try, try again...
            ffmpeg_version = get_ffmpeg_version() # part 2 electric boogaloo
            
        except Exception as e:
            # we should never get here... here be dragons... but... just in case I did, in fact, screw the pooch, let's catch and print the error anyway, _just in case_...
            # this except branch _should_ be unreachable... At this point, we have successfully added the project root ffmpeg bin path to sys.PATH, and we're still not able to get the ffmpeg version
            logging.error(f"\033[31m WTF ERROR: Has Anyone Really Been Far Even As Decided To Use Even Go Want To Do Look More Like?!?! \033[0m")
            logging.error(f"\033[31m ERROR: Could not get ffmpeg version: \033[0m {e}")
            raise e # hot potato with the WTF exception
            sys.exit(1) # fatality: Subzero wins!

        if DEBUG_MODE:
            logging.info(
                f"Outputting _freshly_ updated sys.PATH to ensure our added path is properly set in the sys.PATH:"
            )
            print_formatted_sys_path()

logging.info(
    f"Success: FFMpeg executable found in sys.PATH! Installed FFMpeg version: {ffmpeg_version}"
)


INFO:root:Configuring path for FFMpeg and determining installed FFMpeg version...
INFO:root:Outputting current sys.PATH:
INFO:root: [sys.PATH] 
INFO:root:[
    "D:\\Users\\andrew.sparkes\\AppData\\Roaming\\uv\\python\\cpython-3.12.12-windows-x86_64-none\\python312.zip",
    "D:\\Users\\andrew.sparkes\\AppData\\Roaming\\uv\\python\\cpython-3.12.12-windows-x86_64-none\\DLLs",
    "D:\\Users\\andrew.sparkes\\AppData\\Roaming\\uv\\python\\cpython-3.12.12-windows-x86_64-none\\Lib",
    "D:\\Users\\andrew.sparkes\\AppData\\Roaming\\uv\\python\\cpython-3.12.12-windows-x86_64-none",
    "z:\\code\\STAT405_AudioNotetaker\\.venv",
    "",
    "z:\\code\\STAT405_AudioNotetaker\\.venv\\Lib\\site-packages",
    "\\\\zdrive.labs.cset.oit.edu\\zdrive\\andrew.sparkes\\code\\STAT405_AudioNotetaker\\ffmpeg\\bin",
    "\\\\zdrive.labs.cset.oit.edu\\zdrive\\andrew.sparkes\\code\\STAT405_AudioNotetaker\\ffmpeg\\bin",
    "\\\\zdrive.labs.cset.oit.edu\\zdrive\\andrew.sparkes\\code\\STAT405_AudioNotetaker\\f

NameError: name 'ffmpeg_version' is not defined

In [ ]:
# ensure folders and paths exist
# the models_dir is automatically created regardless, and as of now, output is dumped in cwd, to change later
# sample_data folder doesn't exactly need to be tested for existence for now


In [ ]:
setup_end_time = time.perf_counter()  # end time for setup
setup_duration = setup_end_time - start_time  # in secs, reusing start_time as time marker for the beginning of the setup phase
setup_duration_str = str(timedelta(seconds=setup_duration))  # convert to stringified timedelta HH:MM:SS.mmm format

logging.info(
    f"Imports, initial setup, FFMpeg detection, CUDA detection, hardware spec detection, and remaining configuration options completed in {setup_duration_str}! Current datetime: {datetime.now()}"
)

# next several chunks are mostly wrappers and helpers to wrangle the WhisperX model and associated fucntionality


In [ ]:
# setting up the whisperx model, just a simple wrapper
def load_model(model_size: str, device: str, compute_type: str) -> WhisperModel:
    """Load the WhisperX ASR model."""
    
    logging.info(
        f"[1/4] Loading Whisper model '{model_size}' on {device} ({compute_type})..."
    )
    
    model = whisperx.load_model(
        model_size,
        device=device,
        compute_type=compute_type,
        language=LANGUAGE,
        download_root=MODEL_DIR,
    )
    
    return model


In [ ]:
# bypassing all the issues with torchcodec, using ffmpeg instead to decode audio
def load_audio_via_ffmpeg(file_path: str, sample_rate: int = 16000) -> dict:
    """
    Decode audio using ffmpeg -> numpy -> torch tensor process flow.
    Returns a dict that pyannote/whisperx can accept as pre-loaded audio in format:
        {'waveform': (1, N) torch.Tensor, 'sample_rate': int}
    Bypasses torchcodec entirely. Excellent.
    """

    # don't question the black magic voodoo below, just let ffmpeg do its thing and hand it off to pytorch
    cmd = [
        "ffmpeg",
        "-nostdin",
        "-threads",
        "0",
        "-i",
        file_path,
        "-f",
        "f32le",  # raw 32-bit float PCM, little-endian
        "-ac",
        "1",  # mono
        "-ar",
        str(sample_rate),  # resample to target rate
        "-",  # pipe to stdout
    ]

    result = subprocess.run(cmd, capture_output=True, check=True)
    audio_np = np.frombuffer(result.stdout, dtype=np.float32).copy()  # deep copy
    waveform = torch.from_numpy(audio_np).unsqueeze(0)  # (1, N)

    return {"waveform": waveform, "sample_rate": sample_rate}


In [ ]:
# trying to get anything to callback with progress updates has so far been a complete failure, though the verbose/print_progress flags has helped somewhat
def progress_callback(percent_complete):
    """A simple callback function to output transcription progress updates."""
    
    logging.info(f"Transcription Progress Update: {percent_complete}% completed!")


In [ ]:
# largely a simple wrapper, but this is the core function right here
def transcribe_audio(model, audio_path: str) -> tuple[dict, dict]:
    """Run the initial transcription pass using the ffmpeg-decoded audio."""

    logging.info(f"[2/4] Transcribing '{audio_path}' ...")
    t0 = time.time()

    audio_input = load_audio_via_ffmpeg(audio_path)

    # whisperx.transcribe also accepts the raw numpy array (1-D float32)
    audio_np = audio_input["waveform"].squeeze(0).numpy()

    result = model.transcribe(
        audio_np,
        batch_size=BATCH_SIZE,
        language=LANGUAGE,
        print_progress=True, # progress bar!
    )

    logging.info(
        f"      Done in {time.time() - t0:.1f}s  |  detected language: {result.get('language', 'unknown')}"
    )

    # Return both result and raw audio_input dict (needed for later alignment/diarization steps)
    return result, audio_input


In [ ]:
# word level timestamp alignment
def align_transcript(result: dict, audio: np.ndarray, device: str) -> dict:
    """Align transcript to audio for word-level timestamps."""

    logging.info("[3/4] Aligning transcript (word-level timestamps) ...")
    lang = result.get("language", "en")  # defaults to english if unable to auto-detect

    try:
        align_model, align_metadata = whisperx.load_align_model(
            language_code=lang,
            device=device,
        )

        result_aligned = whisperx.align(
            result["segments"],
            align_model,
            align_metadata,
            audio,
            device,
            return_char_alignments=False,
        )

        return result_aligned

    except Exception as e:
        logging.error(
            f"      \033[31m Recoverable ERROR! Alignment failed: \033[0m {e}. \nDefaulting to using unaligned output."
        )

        return result


In [ ]:
# the hardest bit, diarization of the transcribed audio...
def diarize_transcript(
    result: dict, audio_input: dict, hf_token: str, device: str
) -> dict:
    """Assign speaker labels via pyannote diarization."""

    logging.info(f"[3b/4] Starting diarization (segment speaker identification) phase...")

    try:
        from pyannote.audio import Pipeline

        diarize_model = Pipeline.from_pretrained(
            "pyannote/speaker-diarization-3.1",
            token=hf_token,
            cache_dir=MODEL_DIR
        ).to(torch.device(device))

        diarized_segments = diarize_model(audio_input)

        result = whisperx.assign_word_speakers(diarized_segments, result)

    except Exception as e:
        logging.error(
            f"      \033[31m Recoverable ERROR: Diarization failed: \033[0m {e}. Skipping diarization..."
        )

    return result


In [ ]:
# manipulate default timestamp format here (for transcript timestamps only!):
def format_timestamp(seconds: float) -> str:
    """Convert float seconds to HH:MM:SS.mmm string."""

    ms = int((seconds % 1) * 1000)
    s = int(seconds)
    m, s = divmod(s, 60)
    h, m = divmod(m, 60)

    return f"{h:02d}:{m:02d}:{s:02d}.{ms:03d}"


In [ ]:
# basic transcript output until diarization problem is solved, modify this as needed for desired style of output transcript # TODO may be worth duplicating this function somewhat and/or adding a parameter for output_type to return the transcript in a format that is more easily parsed by other parts of the pipeline downstream (JSON probably is best bet?), whatever form it may take
def build_transcript_text(result: dict) -> str:
    """
    Render speaker segments to a readable plain-text transcript.
    Default format per line:
        [HH:MM:SS.mmm --> HH:MM:SS.mmm]  (SPEAKER_XX)  text
    """

    lines = []

    for seg in result.get("segments", []):
        start = format_timestamp(seg.get("start", 0.0))
        end = format_timestamp(seg.get("end", 0.0))
        text = seg.get("text", "").strip()
        speaker = seg.get("speaker", "")
        speaker_tag = f"  [{speaker}]" if speaker else ""

        lines.append(f"[{start} --> {end}]{speaker_tag}  {text}")

    return "\n".join(lines)


In [ ]:
def save_output(transcript_text: str, result: dict, output_path: str) -> None:
    """Write formatted transcript and a companion JSON file with properly formatted metadata."""

    logging.info(f"[4/4] Writing transcript to '{output_path}'...")

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(transcript_text)
        f.write("\n")  # trailing newline

    # simultaneously save full JSON dump to file (with timestamps, word data, etc) alongside the txt, same filename stem, but with .json extension
    json_path = os.path.splitext(output_path)[0] + ".json"

    # TODO add additional metadata to JSON object before writing to file/object in memory
    
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    logging.info(f"      JSON data saved to '{json_path}'")


In [ ]:
# --- Validate input ---
if not os.path.isfile(INPUT_FILE):
    logging.error(f"\033[31m FATAL ERROR: Input file not found: \033[0m '{INPUT_FILE}'!")
    sys.exit(1)  # fatality: unrecoverable, abort

# --- Pipeline ---
model_load_start_time = time.perf_counter()  # start time for model loading (to measure download speed performance in online mode, and attempt to gauge the speed at which it can actually load the cached local model in offline mode)
model = load_model(MODEL_SIZE, DEVICE, COMPUTE_TYPE)
model_load_end_time = time.perf_counter()  # end time for model loading
model_load_duration_str = str(timedelta(seconds = model_load_end_time - model_load_start_time))
logging.info(f"Initial WhisperX model loaded in {model_load_duration_str}.")


transcription_start_time = time.perf_counter()  # start time for transcription
(result, audio_input) = transcribe_audio(model, INPUT_FILE)
transcription_end_time = time.perf_counter()  # end time for transcription
transcription_duration_str = str(timedelta(seconds = transcription_end_time - transcription_start_time))
logging.info(f"Audio transcription/translation phase completed in {transcription_duration_str}!")


logging.info(f"Freeing up memory by releasing current model from RAM/VRAM...")

# Free model memory before loading align model to conserve already-scarce hardware resources (very helpful on CPU/low-RAM, on low-spec'd devices, this is basically 100% necessary)
del model  # this is super duper important running on mediocre hardware

# word aligned output
if ALIGN_OUTPUT:
    alignment_start_time = time.perf_counter()  # start time for alignment
    result = align_transcript(
        result, audio_input["waveform"].squeeze(0).numpy(), DEVICE
    )
    alignment_end_time = time.perf_counter()  # end time for alignment
    alignment_duration_str = str(timedelta(seconds = alignment_end_time - alignment_start_time))
    logging.info(f"Alignment process completed in {alignment_duration_str}!")

# diarization (segment speaker identification and labeling)
if DIARIZE:
    if not HF_TOKEN: # needs valid HF token to initially download the gated models, after that, can run off cached models offline
        logging.warning(
            f"\033[31m WARNING: DIARIZE=True but HF_TOKEN is not set! \033[0m Skipping diarization process..."
        )

    else:
        logging.info(f"Beginning diarization phase...")
        diarization_start_time = time.perf_counter()  # start time for diarization
        result = diarize_transcript(result, audio_input, HF_TOKEN, DEVICE)
        diarization_end_time = time.perf_counter()  # end time for diarization
        diarization_duration_str = str(timedelta(seconds = diarization_end_time - diarization_start_time))
        logging.info(f"Diarization process completed in {diarization_duration_str}!")


# the following bit will be removed once this code is merged into main, as we won't be writing to a local data file (unnecessary), we can simply keep the transcript JSON object in memory to hand off to the next step in the pipeline
build_and_save_transcript_start_time = time.perf_counter()  # start time for building and saving transcript files to local store
transcript_text = build_transcript_text(result)
save_output(transcript_text, result, OUTPUT_FILE)
build_and_save_transcript_end_time = time.perf_counter()  # end time for building and saving transcript files
build_and_save_transcript_duration_str = str(timedelta(seconds = build_and_save_transcript_end_time - build_and_save_transcript_start_time))
logging.info(f"Building the transcript text from the JSON object and export of data to local txt/json files completed in {build_and_save_transcript_duration_str}!")


logging.info(f"Transcription process fully complete! Current datetime: {datetime.now()}")
logging.info(f"  Text : {OUTPUT_FILE}")
logging.info(f"  JSON : {os.path.splitext(OUTPUT_FILE)[0]}.json")

# despite multiple attempts to suppress the Lightning checkpoint version auto-upgrade part, as well as running the (supposed "permanent" fix) upgrade command as the output suggests, it yet persists... As it says, it automatically upgrades the checkpoint version by itself, so this can safely be ignored and written off as additional annoyingly verbose output


In [ ]:
end_time = time.perf_counter()  # end time for execution
logging.info(f"Execution completed! End time: {datetime.now()}")

execution_duration = end_time - start_time  # in secs
duration_str = str(
    timedelta(seconds=execution_duration)
)  # convert to stringified timedelta HH:MM:SS.mmm format

logging.info(f"Entire (transcription/translation/diarization/alignment steps of the pipeline) process pipeline completed in {duration_str}!")

# initial testing on this lacking hardware laptop, with diarization and alignment both enabled, with online mode enabled to download and cache the required models, using an uncompressed WAV file (duration: ~3:17) in testing resulted in a non-diarized (due to failure in pipeline) word-aligned transcript in approximately 21 minutes. Utilizing that as a rough benchmark, a user with a similarly-powered device would likely expect to see a similar processing time roughly proportional to the duration of the input audio file, approximately 7:1 (in this specific case)in terms of ratio of processing time to initial audio file length. Subsequent testing has been inconsistent and all over the map, highly reactive towards _any_ other applications running that require even tiny amounts of resources, ie. no browsers, no additional VSCode instances, etc. For lowly-spec'd devices, would suggest running this application after a fresh reboot.
# additional tests with various other test audio files (mp4, mp3, etc) resulted in similar computation/processing times.
# projecting from here, a "typical" 50-minute "hour" therapy session would take at minimum, 350 minutes (5 hours, 50 minutes) to transcribe as a lower bound, and many of my failing test files were unable to be diarized properly, so it's a reasonable assumption that we're looking at a baseline minimum processing time of _at least_ ~6 hours for a single 50-minute therapy session when the diarization pipeline is used as well, as in the "normal" use case we've primarily designed for. Despite this significantly lackluster performance, the effect of this level of processing/compute time will be minimized by processing the audio files in the background, and/or to be processed overnight, etc.
# Additional hits to performance likely would be caused by things such as varying (as in multiple languages present in the same audio file) languages used in the input audio, as well as increasing complexity _significantly_ if there are multiple speakers (the diarization pipeline, given our hardware constraints, does not do so great at this), possibly requiring further pre-processing of the audio file into single line or single-language segments before sending off the data down the rest of the processing pipeline. This is attainable, but would complicate things somewhat, and would likely also result in a performance hit due to trading a single (or very few, depending on batch size configuration, etc) call(s) to each model for the entire file for calling a model _for each and every_ segmented line in the file. Current testing results are not promising if the input has multiple languages spoken, significantly more so if each segmented line can be multilingual, as in the case of a native speaker using common cultural expressions or slang in their native language in a single segment/spoken line. This would likely manifest as a moderately-involved refactoring of the pipeline to facilitate this, but would also likely result in a noticeable performance hit by making so many more calls to the resource intensive models.
# performance could be improved by opting to _not_ use the alignment pipeline, and foregoing word-level timestamp alignment, as such detailed identification is not necessarily required for this project.

# language detection accuracy, transcription accuracy, and translation accuracy measures have not yet been implemented as of yet. 
